In [ ]:
import re
import matplotlib.pyplot as plt
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path  = '/content/drive/MyDrive/LM-Info-Pizzo-Davide/DatiMBLIP/'
output_dir = '/content/drive/MyDrive/LM-Info-Pizzo-Davide/graphs/mblip/'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def parse_report(filepath):
    with open(filepath, 'r') as f:
        content = f.read()

    epochs = content.split('--- Epoch')[1:]
    results = []

    for epoch_text in epochs:
        epoch_num   = int(re.search(r'(\d+)', epoch_text).group(1))
        macro_f1    = float(re.search(r'macro avg\s+[\d.]+\s+[\d.]+\s+([\d.]+)', epoch_text).group(1))
        accuracy    = float(re.search(r'accuracy\s+([\d.]+)', epoch_text).group(1))
        not_miso_f1 = float(re.search(r'Not misogynous\s+[\d.]+\s+[\d.]+\s+([\d.]+)', epoch_text).group(1))
        miso_f1     = float(re.search(r'Misogynous\s+[\d.]+\s+[\d.]+\s+([\d.]+)', epoch_text).group(1))

        results.append({
            'epoch': epoch_num,
            'macro_f1': macro_f1,
            'accuracy': accuracy,
            'not_miso_f1': not_miso_f1,
            'miso_f1': miso_f1
        })

    return results

In [ ]:
last_layer = parse_report(base_path + 'report_mblip_last_layer.txt')
full_fine  = parse_report(base_path + 'report_mblip_full_fine.txt')

epochs_last = [r['epoch'] for r in last_layer]
epochs_full = [r['epoch'] for r in full_fine]

In [ ]:
# Macro F1

plt.figure(figsize=(10, 6))
plt.plot(epochs_last, [r['macro_f1'] for r in last_layer],
         marker='o', color='plum', label='Last Layer', linewidth=2)
plt.plot(epochs_full, [r['macro_f1'] for r in full_fine],
         marker='s', color='gold', label='Q-Former Fine-Tuned', linewidth=2)
plt.axhline(y=0.69, color='blue', linestyle='--', alpha=0.5, label='mCLIP Fixed Features (0.69)')
plt.axhline(y=0.68, color='green', linestyle='--', alpha=0.5, label='mCLIP Full Fine-Tuned (0.68)')
plt.title('Macro F1 per epoca', fontsize=16, fontweight='bold')
plt.xlabel('Epoca', fontsize=14)
plt.ylabel('Macro F1', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig(output_dir + 'macro_f1.png', dpi=300)
plt.show()

In [ ]:
# accuracy

plt.figure(figsize=(10, 6))
plt.plot(epochs_last, [r['accuracy'] for r in last_layer],
         marker='o', color='plum', label='Last Layer', linewidth=2)
plt.plot(epochs_full, [r['accuracy'] for r in full_fine],
         marker='s', color='gold', label='Q-Former Fine-Tuned', linewidth=2)
plt.title('Accuracy per epoca', fontsize=16, fontweight='bold')
plt.xlabel('Epoca', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig(output_dir + 'accuracy.png', dpi=300)
plt.show()

In [ ]:
#F1 per class Last Layer

plt.figure(figsize=(10, 6))
plt.plot(epochs_last, [r['not_miso_f1'] for r in last_layer],
         marker='o', color='blue', label='Not Misogynous', linewidth=2)
plt.plot(epochs_last, [r['miso_f1'] for r in last_layer],
         marker='s', color='red', label='Misogynous', linewidth=2)
plt.title('F1 per classe — Last Layer Fine-Tuned', fontsize=16, fontweight='bold')
plt.xlabel('Epoca', fontsize=14)
plt.ylabel('F1-score', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig(output_dir + 'f1_classes_last.png', dpi=300)
plt.show()

In [ ]:
#F1 per class Full-fine(o Q-former fine tuned)
plt.figure(figsize=(10, 6))
plt.plot(epochs_full, [r['not_miso_f1'] for r in full_fine],
         marker='o', color='blue', label='Not Misogynous', linewidth=2)
plt.plot(epochs_full, [r['miso_f1'] for r in full_fine],
         marker='s', color='red', label='Misogynous', linewidth=2)
plt.title('F1 per classe — Q-Former Fine-Tuned', fontsize=16, fontweight='bold')
plt.xlabel('Epoca', fontsize=14)
plt.ylabel('F1-score', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig(output_dir + 'f1_classes_full.png', dpi=300)
plt.show()